- Python에서 비동기 처리는 하나의 스레드에서 작동하며,
- Non-blocing 코드를 만났을 때 Task Queue(event loop)에 있는 다른 Task로 넘어갑니다.
    - blocking 코드란, 해당 코드가 실행이 완료될 때 까지 스레드 진행을 멈추게 하는 코드입니다. 
        - 예를 들어, requests 요청 코드는 응답이 올 때까지 스레드를 대기 상태로 들어가게 합니다. 그리고 응답이 도착했을 때 OS가 해당 프로세스(스레드)를 깨워서 준비 상태로 전환시키고 실행 상태일 때 다음 코드로 넘어갈 수 있게 됩니다. 
    non-blocking 코드는 반대로, 스레드가 대기 상태로 넘어가지 않고, 스레드의 다음 코드를 실행하도록 합니다.
        - httpx.AsyncClient 비동기 코드를 이용해서 HTTP 요청을 보냈을 때, 다른 스레드로 전환되지 않고 현재 스레드에서 Event Loop에 의해 다른 Task를 실행시킵니다.
- Coroutine은 실행할 비동기 코드를 의미하며, Event Loop에 의해 Task로 변환되고 실행됩니다.
- Task는 논리 스레드 단위이며, Task에 대한 Context는 힙 메모리에서 관리됩니다.
- Coroutine 내부에 Coroutine이 있더라도, 바깥 쪽에서 asyncio.create_task에 의해 Task로 만들 때 2개의 Task로 생성되지 않고 하나의 Task로 생성됩니다.
- Task 코드에서 Exception 예외 처리가 있어도, 다른 Task에 영향을 주지 않습니다. (리소스 사용 제외)
- 멀티스레딩과 기능 및 역할이 비슷해보입니다.
    - 멀티스레딩에서는 Blocking 코드를 만났을 때, 다른 스레드의 코드를 실행하도록 합니다. (blocking 코드가 없더라도 일정 시간 마다 context switching이 일어납니다.)
- 하지만, 비동기 프로세스에서는 kernel 영역에서의 context switching이 일어나지 않아, 보다 효율적으로 작동합니다.
- 디스크 I/O 같은 Blocking 코드는 asyncio.to_thread를 통해 새로운 스레드를 만들어(또는 ThreadPool의 기존 스레드를 이용하여) non-blocking 코드처럼 실행시킬 수 있습니다.
- ContextVar를 이용해서 Task 영역에서 사용할 데이터를 따로 선언하고 관리 가능합니다.

In [9]:
import asyncio
import time


async def non_blocking_task(x, blocking_time=0, latency=3):
    print(f"non_blocking_task {x}")
    time.sleep(blocking_time) # Blocking Code(or CPU-bound code)
    
    print(f"non_blocking_task {x} sleep {latency}")
    await asyncio.sleep(latency)
    print(f"non_blocking_task {x} done")

async def main():
    # 1. 첫번째 비동기 코드가 시작되고, latency 대기하게 됩니다.
    # 2. 첫번째 비동기 코드가 대기하게 될 때, 두번째 비동기 코드가 실행됩니다.
    # 3. 첫번째 비동기 코드의 응답이 도착했어도, 아직 두번째 비동기 코드가 실행되고 있습니다.
    # 4. 두번째 비동기 코드가 대기 상태로 들어갈 때, 첫번째 비동기 코드가 실행됩니다.
    # 5. 두번째 비동기 코드 응답이 도착 후, 다음 task가 없으므로 바로 두번째 비동기 코드가 실행됩니다.
    await asyncio.gather(
        non_blocking_task(1, 2, 1),
        non_blocking_task(2, 4, 2),
    )

await main()


non_blocking_task 1
non_blocking_task 1 sleep 1
non_blocking_task 2
non_blocking_task 2 sleep 2
non_blocking_task 1 done
non_blocking_task 2 done


In [16]:
from contextvars import ContextVar

outer_var = ContextVar('outer_var')
inner_var = ContextVar('inner_var')

# 바깥쪽의 비동기 코드(코루틴)에서 실행되는 안쪽 비동기 코드는 바깥쪽 비동기 코드의 ContextVar을 공유합니다.

async def outer():
    outer_var.set('outer set')
    print(outer_var.get())
    await inner()
    print(outer_var.get())
    
    print(inner_var.get())
    inner_var.set('inner changed')
    print(inner_var.get())

async def inner():
    print(outer_var.get())
    outer_var.set('outer changed')
    print(outer_var.get())

    inner_var.set('inner set')
    print(inner_var.get())
await outer()

outer set
outer set
outer changed
inner set
outer changed
inner set
inner changed


In [30]:
import threading

global_var = 'global'
thread_local = threading.local()
var1 = ContextVar('var1')

# asyncio.gather에 의해 실행되는 비동기 코드는 각각 다른 Task로 실행되고, 각각의 Task는 다른 Task의 ContextVar을 공유하지 않습니다.
# 반면, 전역 변수나 Thread.local()에 의해 생성된 변수는 모든 Task에서 공유됩니다.
async def task1():
    global_var = 'global: task1'
    print(global_var)
    thread_local.var1 = 'task1_thread_local'
    var1.set('task1 set')
    print(var1.get())

async def task2():
    print(global_var)
    print(thread_local.var1)
    print(var1.get())
    

await asyncio.gather(task1(), task2())

global: task1
task1 set
global
task1_thread_local


/Users/ibird/workspace/playground/.venv/lib/python3.12/site-packages/pygments/regexopt.py:78: RuntimeWarning: coroutine 'main' was never awaited
  for group in groupby(strings, lambda s: s[0] == first[0])) \


LookupError: <ContextVar name='var1' at 0x10ae5f6f0>

In [38]:
async def waiter(event):
    print('waiting for it ...')
    await event.wait()
    print('... got it!')

async def main():
    # Event 객체를 만듭니다.
    event = asyncio.Event()

    # 'event'가 설정될 때까지 대기 할 Task를 만듭니다.
    waiter_task = asyncio.create_task(waiter(event))

    # 1초 동안 잠잔 후에 event를 설정합니다. 이 때, 대기 중인 waiter 태스크가 실행되고, event.wait()에서 대기 상태로 들어갑니다.
    await asyncio.sleep(1)
    
    # event를 설정합니다. 이 때, waiter 태스크는 준비 상태로 전환되었지만, 아직 main 태스크가 끝나지 않아서 실행되지 않습니다.
    event.set()

    # waiter의 나머지 task가 실행됩니다
    await waiter_task

await main()

waiting for it ...
... got it!


In [40]:
async def task1():
    print("task1 start")

    async with asyncio.Lock():
        print("task1 lock")
        await asyncio.sleep(3)
        print("task1 unlock")

    print("task1 end")

async def task2():
    print("task2 start")
    async with asyncio.Lock():
        print("task2 lock")
        await asyncio.sleep(1)
        print("task2 unlock")

    print("task2 end")

await asyncio.gather(task1(), task2())


task1 start
task1 lock
task2 start
task2 lock
task2 unlock
task2 end
task1 unlock
task1 end


[None, None]

In [ ]:
# asyncio.Lock은 비동기 프로세스에서의 Mutex Lock으로, Lock 영역에서의 코드 및 변수를 하나의 Task에서만 실행할 수 있도록 한다.
last_seen = {}
async_lock = asyncio.Lock()

async def task1():
    print("task1 start")
    async with async_lock:
        print("task1 lock")
        await asyncio.sleep(2)
        print("task1 unlock")
    print("task1 end")

async def task2():
    print("task2 start")
    async with async_lock:
        print("task2 lock")
        await asyncio.sleep(2)
        print("task2 unlock")
    print("task2 end")

await asyncio.gather(task1(), task2())


task1 start
task1 lock
task2 start
task1 unlock
task1 end
task2 lock
task2 unlock
task2 end


[None, None]

In [52]:
# asyncio.to_thread는 동기 코드를 다른 스레드에서 실행하도록 하여, 비동기 코드 처럼 실행할 수 있도록 합니다.
import threading

def io_bound_blocking_code():
    print(f"io_bound_blocking_code start in {threading.get_ident()}")
    time.sleep(2)
    print(f"io_bound_blocking_code end in {threading.get_ident()}")

async def task1():
    print(f"task1 start in {threading.get_ident()}")
    await asyncio.to_thread(io_bound_blocking_code)
    print(f"task1 end in {threading.get_ident()}")

async def task2():
    print(f"task2 start in {threading.get_ident()}")
    await asyncio.sleep(1)
    print(f"task2 end in {threading.get_ident()}")
    

await asyncio.gather(task1(), task2())


task1 start in 8649941184
task2 start in 8649941184
io_bound_blocking_code start in 6259552256
task2 end in 8649941184
io_bound_blocking_code end in 6259552256
task1 end in 8649941184


[None, None]

In [ ]:
async def task1():
    print("task1 start")
    await asyncio.sleep(1)
    print("task1 end")


async def task2():
    print("task2 start")
    raise Exception("task2 error")
    print("task2 end")


async def task3():
    print("task3 start")
    await asyncio.sleep(1)
    print("task3 end")


await asyncio.gather(task1(), task2(), task3())

task1 start
task2 start
task3 start


Exception: task2 error

task1 end
task3 end


In [54]:
async def eternity():
    # Sleep for one hour
    await asyncio.sleep(3600)
    print('yay!')

async def main():
    # Wait for at most 1 second
    try:
        await asyncio.wait_for(eternity(), timeout=1.0)
    except asyncio.TimeoutError:
        print('timeout!')

await main()

timeout!
